In [ ]:
import numpy as np
import pandas as pd
import re
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Compositional Language and Elementary Visual Reasoning (CLEVR)

Objektuek bi motatako erlazioak dute, geometrikoa eta konparaziozko logika:


1.   **Spatial Relationships.** `left`, `right`, `front`, `behind`.
2.  **Same-Attribute Relationships.** `same_shape`, `same_size`, `same_material`, `same_color`.



# Irudiak

In [ ]:
from PIL import Image

Irudien banaketa:


| TRAIN | VAL | TEST |
|--------------|--------------|--------------|
| 70K| 15K | 15K |




In [ ]:
TRAIN_JSON = '/content/drive/MyDrive/TFG/CLEVR_train_scenes.json'
VALID_JSON = '/content/drive/MyDrive/TFG/CLEVR_val_scenes.json'

In [ ]:
with open(TRAIN_JSON, 'r') as f:
    train_data = json.load(f)

with open(VALID_JSON, 'r') as f:
    valid_data = json.load(f)

train_scenes_count = len(train_data['scenes'])
valid_scenes_count = len(valid_data['scenes'])

print(f"Total scenes in TRAIN_JSON: {train_scenes_count}")
print(f"Total scenes in VALID_JSON: {valid_scenes_count}")
print(f"Total elements (train + valid): {train_scenes_count + valid_scenes_count}")

Total scenes in TRAIN_JSON: 70000
Total scenes in VALID_JSON: 15000
Total elements (train + valid): 85000


## JSON fitxategia

JSON fitxategiko edukia, irudi bakoitzarentzat:



1.   **Objects list:** `color`, `size`, `rotation`, `shape`, `3D coords` (real world), `material`, `pixel coords` (2D projection XYZ, kamarekiko erlatiboak). XYZ koordenatuek objektuaren zentro geometrikoa adierazten dute.

  *   **3D koordenatuek** objektuek CLEVR-ean simulatutako mundu birtualean duten posizio erreala adierazten dute, non (0,0,0) jatorria eszenaren zentroa den. X ezker-eskuin, Y aurre.atze (sakonera), eta Z-k objektuaren zentroa (`h/2`).
  *   **Pixel koordenatuek** objektuak 2D irudian proiektatutako posizioa adierazten dute.
2.   **Directions:** `left`, `right`, `front`, `behind`, `above`, `below`. Kamararen posizio eta orientazioa adierazten duten norabide bektoreak dira.

3.   **Relationships:** `left`, `right`, `front`, `behind`. Esaterako: relationships['right'][0] = [2, 5] denean, 0 objektuak 2 eta 5 objektuak ditu bere eskuinean.

In [ ]:
with open(TRAIN_JSON, 'r') as f:
    data = json.load(f)

for scene in data['scenes'][:1]:
    basic_keys = {k: v for k, v in scene.items() if k != "relationships"}
    print(json.dumps(basic_keys, indent=2, separators=(', ', ': ')))

    print('"relationships": {')
    for rel_name, rel_lists in scene['relationships'].items():
        print(f'  "{rel_name}": [')
        for sublist in rel_lists:
            print(f'    {sublist},')
        print('  ],')
    print('}')

{
  "image_index": 0, 
  "objects": [
    {
      "color": "blue", 
      "size": "large", 
      "rotation": 269.8517172617167, 
      "shape": "cube", 
      "3d_coords": [
        -1.3705521821975708, 
        2.0794010162353516, 
        0.699999988079071
      ], 
      "material": "rubber", 
      "pixel_coords": [
        269, 
        88, 
        12.661545753479004
      ]
    }, 
    {
      "color": "green", 
      "size": "large", 
      "rotation": 292.2219458666971, 
      "shape": "cylinder", 
      "3d_coords": [
        -2.9289753437042236, 
        -1.7488206624984741, 
        0.699999988079071
      ], 
      "material": "metal", 
      "pixel_coords": [
        93, 
        108, 
        11.522202491760254
      ]
    }, 
    {
      "color": "cyan", 
      "size": "small", 
      "rotation": 25.545135239473026, 
      "shape": "cube", 
      "3d_coords": [
        1.5515961647033691, 
        0.6776641607284546, 
        0.3499999940395355
      ], 
      "materia

In [ ]:
with open('/content/data_split.json', 'r') as f:
    data_split = json.load(f)

with open(TRAIN_JSON, 'r') as f:
    original_train_data = json.load(f)

with open(VALID_JSON, 'r') as f:
    original_valid_data = json.load(f)

# Zenbat eszena train, val eta testean?
new_train_scenes = [original_train_data['scenes'][idx] for idx in data_split['train']]
new_train_data = {'scenes': new_train_scenes, 'info': original_train_data['info']}

new_valid_scenes = [original_train_data['scenes'][idx] for idx in data_split['val']]
new_valid_data = {'scenes': new_valid_scenes, 'info': original_train_data['info']}

new_test_data = {'scenes': original_valid_data['scenes'], 'info': original_valid_data['info']}

print(f"New train scenes count: {len(new_train_data['scenes'])}")
print(f"New valid scenes count: {len(new_valid_data['scenes'])}")
print(f"New test scenes count: {len(new_test_data['scenes'])}")

New train scenes count: 59500
New valid scenes count: 10500
New test scenes count: 15000


# 3D Voxel Generator

In [ ]:
def generate_3d_voxel(scene, grid_size):

    # 1. Munduaren koordenatuen limiteak
    limits_x = (-4, 4)
    limits_y = (-4, 4)
    limits_z = (0.0, 1.4)

    x_range = np.linspace(limits_x[0], limits_x[1], grid_size)
    y_range = np.linspace(limits_y[0], limits_y[1], grid_size)
    z_range = np.linspace(limits_z[0], limits_z[1], grid_size)

    # 2. `grid_size` tamaina duen grid-aren (X, Y, Z) guztiak gordetzen dira
    gx, gy, gz = np.meshgrid(x_range, y_range, z_range, indexing='ij')

    voxels = np.zeros((grid_size, grid_size, grid_size), dtype=np.uint8)

    # 3. Nola jakin puntu bat objektu baten parte den?
    for obj in scene['objects']:

        # Objektuaren zentroa
        xc, yc, zc = obj['3d_coords']
        r = 0.35 if obj['size'] == 'small' else 0.7
        theta = np.radians(obj.get('rotation', 0))

        # Zein distantziatara dago puntu bakoitza objektuaren zentrora?
        dx, dy, dz = gx - xc, gy - yc, gz - zc

        if obj['shape'] == "sphere":
            mask = (dx**2 + dy**2 + dz**2) <= r**2
        elif obj['shape'] == "cube":
            dx_rot = dx * np.cos(theta) + dy * np.sin(theta)
            dy_rot = -dx * np.sin(theta) + dy * np.cos(theta)
            mask = (np.abs(dx_rot) <= r) & (np.abs(dy_rot) <= r) & (np.abs(dz) <= r)
        elif obj['shape'] == "cylinder":
            mask = (dx**2 + dy**2 <= r**2) & (np.abs(dz) <= r)

        voxels[mask] = 1

    return voxels

### Voxela bistaratzen

In [ ]:
import plotly.graph_objects as go

def visualize_grid(voxel_grid):

    limits_x = (-4, 4)
    limits_y = (-4, 4)
    limits_z = (0.0, 1.4)

    grid_size = voxel_grid.shape[0]

    step_x = (limits_x[1] - limits_x[0]) / grid_size
    step_y = (limits_y[1] - limits_y[0]) / grid_size
    step_z = (limits_z[1] - limits_z[0]) / grid_size

    x_idx, y_idx, z_idx = np.where(voxel_grid > 0)

    x_real = limits_x[0] + (x_idx + 0.5) * step_x
    y_real = limits_y[0] + (y_idx + 0.5) * step_y
    z_real = limits_z[0] + (z_idx + 0.5) * step_z

    fig = go.Figure(data=[go.Scatter3d(
        x=x_real, y=y_real, z=z_real,
        mode='markers', marker=dict(size=5, symbol='square', color='blue', opacity=1.0)
    )])


    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X', range=limits_x),
            yaxis=dict(title='Y', range=limits_y),
            zaxis=dict(title='Z', range=limits_z),
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=0.2)
        ),
        margin=dict(r=0, l=0, b=0, t=0)
    )

    fig.show()

### Main

In [ ]:
json_path = TRAIN_JSON
with open(json_path, 'r') as f:
    scene_data = json.load(f)

voxel_grid = generate_3d_voxel(scene_data['scenes'][0], 64)

total_voxels = voxel_grid.size
occupied_voxels = np.sum(voxel_grid > 0)
empty_voxels = total_voxels - occupied_voxels

print(f"Total voxels: {total_voxels}")
print(f"Occupied voxels: {occupied_voxels}, empty voxels: {empty_voxels}")
print(f"Shape: {voxel_grid.shape}")

visualize_grid(voxel_grid)

Total voxels: 262144
Occupied voxels: 25454, empty voxels: 236690
Shape: (64, 64, 64)
